# Computational Analysis of Sounds and Music (CH-CASM-M)

## 07 - Rhythmic Analysis

**WS 2025/2026**

Prof. Dr. Jakob Abeßer, jakob.abesser@uni-bamberg.de

Last update: 03.12.2025

**Outline**



The notebook is inspired by the great ISMIR 2021 Tutorial "Tempo, Beat, and Downbeat Estimation" held by Matthew E. P. Davies, Sebastian Böck, and Magdalena Fuentes. The complementary Jupyter book can be found here:

https://tempobeatdownbeat.github.io/tutorial/intro.html

In our notebook, you will learn how to implement a **beat tracking** system and estimate the **tempo** of a song.

## Preparation

In [ ]:
!pip install wget

In [ ]:
import glob
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as pl
import IPython.display as ipd
import wget
import seaborn as sns

import zipfile

## Dataset

We use in this notebook we will re-use two audio recordings from our previous notebooks.

In [ ]:
fn_list = ['easy_example.beats',
           'easy_example.beats.txt',
           'easy_example.flac',
           'nonwestern_example.beats',
           'nonwestern_example.beats.txt',
           'nonwestern_example.flac',
           '257993__orangefreesounds__disco-funky-beat.wav']

if any([not os.path.isfile(_) for _ in fn_list]):
    for fn in fn_list:
        wget.download('https://github.com/CHBamberg/CH-CASM-M-2025/raw/refs/heads/main/data/{}'.format(fn), 
                      out=fn, bar=None)
        print(f"Downloaded {fn}")
else:
    print('Files already exist!')

In [ ]:
FIGSIZE = (10,2.5)
SR = 44100

In [ ]:
x1, sr = librosa.load("easy_example.flac", sr = SR)
pl.figure(figsize=FIGSIZE)
librosa.display.waveshow(x1, sr=SR, alpha=0.6); # this semi-colon surpresses the matplotlib stdout <matplotlib.zzz.zzz at 0x etc.>
pl.title('Audio waveform', fontsize=15)
pl.yticks(fontsize=12)
pl.xticks(fontsize=12)
pl.xlabel('Time', fontsize=13)
pl.xlim(0, len(x1)/sr);
ipd.Audio(x1, rate=SR) 

### Import beat annotations

Let's import a text file with the annotated beat positions (in seconds).

In [ ]:
beats1 = np.loadtxt('easy_example.beats')
downbeats1 = beats1[beats1[:, 1] == 1][:, 0]
beats1 = beats1[:,0]
print(f"Beat positions: {beats1}")
print(f"Down-beat positions: {downbeats1}")

**Observation**: The first beat is a down-beat, the fivth one, and so forth. This implies that the song has a 4/4 time signature where the beat "1" in each bar is a down-beat.

Let's visualize the waveform again with additional vertical lines indicating the click positions:

In [ ]:
pl.figure(figsize=FIGSIZE)
librosa.display.waveshow(x1, sr=SR, alpha=0.6)
pl.vlines(beats1, 1.1*x1.min(), 1.1*x1.max(), label='Beats', color='r', linestyle=':', linewidth=2)
pl.vlines(downbeats1, 1.1*x1.min(), 1.1*x1.max(), label='Downbeats', color='black', linestyle='--', linewidth=2)
pl.legend(fontsize=12); 
pl.title('Audio waveform with beats and downbeats', fontsize=15)
pl.yticks(fontsize=12)
pl.xticks(fontsize=12)
pl.xlabel('Time', fontsize=13)
pl.xlim(0, len(x1)/sr);

The **librosa** Python module allows to sonify these beats by creating an audio sample vector with a click sound played at the beat times. The ```librosa.clicks()``` method allows to create clicks with higher frequencies (down-beats) and lower frequency (beats). 

In [ ]:
y_beats1 = librosa.clicks(times=beats1, sr=SR, click_freq=1000.0, click_duration=0.1, click=None, length=len(x1))
y_downbeats1 = librosa.clicks(times=downbeats1, sr=SR, click_freq=1500.0, click_duration=0.15, click=None, length=len(x1))

# We can create a mixed audio by mixing the original audio with the two sonifications
ipd.Audio(0.7*x1+0.2*y_beats1+0.2*y_downbeats1, rate=SR) 

## Second example: Non-western music

Here's an audio excerpt of Uruguayan Candombe (taken from the [Candombe Recordings Dataset.](http://www.eumus.edu.uy/candombe/datasets/ISMIR2015/))

In [ ]:
beats2 = np.loadtxt('nonwestern_example.beats')
downbeats2 = beats2[beats2[:, 1] == 1][:, 0]
beats2 = beats2[:,0]
print(f"Beat positions: {beats2}")
print(f"Down-beat positions: {downbeats2}")

x2, sr = librosa.load("nonwestern_example.flac", sr = SR)

In [ ]:
pl.figure(figsize=FIGSIZE)
librosa.display.waveshow(x2, sr=SR, alpha=0.6)
pl.vlines(beats1, 1.1*x2.min(), 1.1*x2.max(), label='Beats', color='r', linestyle=':', linewidth=2)
pl.vlines(downbeats1, 1.1*x2.min(), 1.1*x2.max(), label='Downbeats', color='black', linestyle='--', linewidth=2)
pl.legend(fontsize=12); 
pl.title('Audio waveform with beats and downbeats', fontsize=15)
pl.yticks(fontsize=12)
pl.xticks(fontsize=12)
pl.xlabel('Time', fontsize=13)
pl.xlim(0, len(x2)/sr);

**Task**: Try to tap along, again it's a 4/4 meter.

In [ ]:
# Original
ipd.Audio(x2, rate=SR) 

In [ ]:
# Now again with the sonified beats and down-beats
y_beats2 = librosa.clicks(times=beats2, sr=SR, click_freq=1000.0, click_duration=0.1, click=None, length=len(x2))
y_downbeats2 = librosa.clicks(times=downbeats2, sr=SR, click_freq=1500.0, click_duration=0.15, click=None, length=len(x2))
ipd.Audio(0.7*x2+0.2*y_beats2+0.2*y_downbeats2, rate=SR) 

## Simple Beat-Tracking System based on Spectral Flux

In the following, we will compute a Mel spectrogram from our audio files and compute the spectral flux function, which indicates the amount of change from one time frame to the next one.  

In [ ]:
fps = 100
sr = 44100
n_fft = 2048
hop_length = int(librosa.time_to_samples(1./fps, sr=sr))
n_mels = 80
fmin = 27.5
fmax = 17000.
lag = 2
max_size = 3

# make the mel spectrogram
S = librosa.feature.melspectrogram(x1, sr=sr, n_fft=n_fft,
                                   hop_length=hop_length,
                                   fmin=fmin,
                                   fmax=fmax,
                                   n_mels=n_mels)

In [ ]:
spectral_flux = librosa.onset.onset_strength(S=librosa.power_to_db(S, ref=np.max),
                                      sr=sr,
                                      hop_length=hop_length,
                                      lag=lag, max_size=max_size)

frame_time = librosa.frames_to_time(np.arange(len(spectral_flux)),
                                    sr=sr,
                                    hop_length=hop_length)

In [ ]:
fig, ax = pl.subplots(nrows=3, sharex=True, figsize=(14,6))

librosa.display.waveshow(x1, sr=sr, alpha=0.6, ax=ax[0])

ax[0].set_title('Easy Example: audio waveform')
ax[0].label_outer()

librosa.display.specshow(librosa.power_to_db(S, ref=np.max),
                         y_axis='mel', x_axis='time', sr=sr,
                         hop_length=hop_length, fmin=fmin, fmax=fmax, ax=ax[1])

ax[1].set_title('Mel Spectrogram')
ax[1].label_outer()

ax[2].plot(frame_time, spectral_flux, label='Spectral flux')
ax[2].set_title('Spectral flux')

## Periodicity detection

Based on the spectral flux, we now aim to detect periodicities, which can indicate the tempo of the piece, which is commonly given in **beats per minute** (BPM).

**Autocorrelation** is one of the most commonly used methods for estimating beat periodicity in music signals. In simple terms, it involves creating a duplicate of the signal and sliding it against the original. By identifying the time delays (or lags) at which the two versions resemble each other, we obtain peaks in the autocorrelation function that reveal the underlying rhythmic structure.

The following function 
 - computes the **tempogram** (see https://librosa.org/doc/0.9.2/generated/librosa.beat.tempo.html) 
 - estimates the **tempo** (taken from https://tempobeatdownbeat.github.io/tutorial/ch2_basics/baseline.html)

In [ ]:
def periodicity_estimation_plots(oenv, sr, hop_length, ref_beats=None):
    # first compute the tempogram (
    tempogram = librosa.feature.tempogram(onset_envelope=oenv, sr=sr,
                                          hop_length=hop_length)
    # Compute global onset autocorrelation
    ac_global = librosa.autocorrelate(oenv, max_size=tempogram.shape[0])
    ac_global = librosa.util.normalize(ac_global)
    tempo = librosa.beat.tempo(onset_envelope=oenv, sr=sr, hop_length=hop_length)[0]


    fig, ax = pl.subplots(nrows=4, figsize=(14, 14))
    times = librosa.times_like(oenv, sr=sr, hop_length=hop_length)
    ax[0].plot(times, oenv)
    ax[0].set_title('Spectral flux',fontsize=15)
    ax[0].label_outer()
    ax[0].set(xlim=[0, len(oenv)/fps]);
    librosa.display.specshow(tempogram, sr=sr, hop_length=hop_length,
                             x_axis='time', y_axis='tempo', cmap='magma',
                             ax=ax[1])
    ax[1].axhline(tempo, color='w', linestyle='--', alpha=1,
                label='Estimated tempo={:g}'.format(tempo))
    ax[1].legend(loc='upper right')
    ax[1].set_title('Tempogram',fontsize=15)
    x = np.linspace(0, tempogram.shape[0] * float(hop_length) / sr,
                    num=tempogram.shape[0])
    ax[2].plot(x, np.mean(tempogram, axis=1), label='Mean local autocorrelation')
    ax[2].plot(x, ac_global, '--', alpha=0.75, label='Global autocorrelation')
    ax[2].set(xlabel='Lag (seconds)')
    ax[2].legend(frameon=True)
    freqs = librosa.tempo_frequencies(tempogram.shape[0], hop_length=hop_length, sr=sr)
    ax[3].semilogx(freqs[1:], np.mean(tempogram[1:], axis=1),
                 label='Mean local autocorrelation') #, basex=2)
    ax[3].semilogx(freqs[1:], ac_global[1:], '--', alpha=0.75,
                 label='Global autocorrelation') #, basex=2)
    ax[3].axvline(tempo, color='black', linestyle='--', alpha=.8,
                label='Estimated tempo={:g}'.format(tempo))
    
    if ref_beats is not None:
        gt_tempo = 60./np.median(np.diff(ref_beats))
        ax[3].axvline(gt_tempo, color='red', linestyle='--', alpha=.8,
                label='Tempo derived from beat annotations={:g}'.format(gt_tempo))

    ax[3].legend(frameon=True)
    ax[3].legend(loc='upper right')
    ax[3].set(xlabel='BPM')
    ax[3].grid(True)
    return tempo

In [ ]:
tempo = periodicity_estimation_plots(oenv=spectral_flux, sr=sr, hop_length=hop_length)

## Non-western example

In [ ]:
# make the mel spectrogram
S = librosa.feature.melspectrogram(x2, sr=sr, n_fft=n_fft,
                                   hop_length=hop_length,
                                   fmin=fmin,
                                   fmax=fmax,
                                   n_mels=n_mels)

spectral_flux = librosa.onset.onset_strength(S=librosa.power_to_db(S, ref=np.max),
                                      sr=sr,
                                      hop_length=hop_length,
                                      lag=lag, max_size=max_size)

frame_time = librosa.frames_to_time(np.arange(len(spectral_flux)),
                                    sr=sr,
                                    hop_length=hop_length)

tempo = periodicity_estimation_plots(oenv=spectral_flux, sr=sr, hop_length=hop_length)